# IAU Projekt - fáza 3
**Marek Kujan Privrel** a **Martin Benko** (rozdelenie práce 50:50)  
Cvičenie: Utorok 16:00 s Ing. Laurou Krajčovičovou  

In [44]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.stats as sm_stats
import sklearn
import category_encoders as ce

import math

In [ ]:
train_df = pd.read_csv("train_preprocessed.csv")

test_df = pd.read_csv("test_preprocessed.csv")


In [ ]:
class Node():
    def __init__(self,
            data: pd.DataFrame,
        ):
        self.final = False
        self.data = data
        self.entropy = self.calc_entropy(self.data)
        self.left_child = None
        self.right_child = None
        self.atribute = None
        self.split_point = None

    def calc_entropy(self, data):  # môže vrátiť NaN ak p0 alebo p1 výjdu 0 
        p0, p1 = data["failure"].value_counts(normalize=True).sort_index(ascending=True)
        return - p0 * math.log2(p0) - p1 * math.log2(p1)

    def pick_best_attribute(self) -> tuple:
        cols = self.data.columns.to_list()
        cols.pop(cols.index("failure"))

        best_ig = 0
        best_attr = None
        best_split_point = None

        for col in cols:
            ig, split_point = self.try_attribute(col)
            if ig > best_ig:
                best_ig = ig
                best_attr = col
                best_split_point = split_point

        return best_attr, best_split_point

    def live_and_make_children(self):
        self.atribute, self.split_point = self.pick_best_attribute()
        self.left_child = Node(self.data.loc[self.data[self.atribute <= self.split_point]])
        self.right_child = Node(self.data.loc[self.data[self.atribute > self.split_point]])
        self.left_child.live_and_make_children()
        self.right_child.live_and_make_children()

            
    def try_attribute(self, attribute: str) -> tuple:
        selection = self.data[attribute]
        info_gain = 0
        best_split_point = None

        if selection.nunique() <= 100:
            # spracujeme ako kategorický atribút
            categories = pd.DataFrame(selection.value_counts(normalize=True))

            for cat in categories.index.to_list():
                categories.loc[cat, "entropy"] = self.calc_entropy(self.data.loc[self.data[attribute] == cat])

            categories["product"] = categories.entropy * categories.proportion
            
            weighted_entropy = abs(np.sum(categories.product))
            info_gain = self.entropy - weighted_entropy

            
        else:
            # vypočíta info_gain pre všetky prahy a nájde ten najlepší

            temp = self.data.loc[[attribute, "failure"]].sort_values(by=attribute)
            temp["failure_diff"] = temp["failure"].diff()
            temp['middle'] = (temp[attribute] - temp[attribute].shift(1))/2

            thresholds = temp.loc[temp["failure_diff"] != 0, "middle"]

            thresholds = pd.DataFrame({"threshold": thresholds})

            for threshold in thresholds:
                thresholds["percentage_left"] = temp.loc[attribute <= threshold].size
                thresholds["percentage_right"] = temp.size - thresholds["percentage_left"]

                thresholds["entropy_left"] = self.calc_entropy(temp.loc[attribute <= threshold])
                thresholds["entropy_right"] = self.calc_entropy(temp.loc[attribute > threshold])


            thresholds["weighted_entropy"] = (thresholds["percentage_left"]*thresholds["entropy_left"]
                                    + thresholds["percentage_right"]*thresholds["entropy_right"])

            min = thresholds["weighted_entropy"].idxmin()
            info_gain = self.entropy - thresholds.at[min, "weighted_entropy"]
            best_split_point = thresholds.at[min, "threshold"]            

        return info_gain, best_split_point


    


In [52]:

class Node():
    def __init__(self, data: pd.DataFrame):
        self.final = False                              # či je tento uzol posledný/rozhodujúci
        self.data = data                                # dataframe, s ktorým tu pracujeme
        self.entropy = self.calc_entropy(self.data)     # entropia tohto uzla
        self.left_child = None                          # ľavá vetva
        self.right_child = None                         # pravá vetva
        self.atribute = None                            # názov atribútu, podľa ktorého sa v tomto uzle rozhoduje
        self.split_point = None             # pre numerické atribúty číselná hranica, s ktorou sa atribút porovná
        self.prediction = None              # predikcia failure v tomto uzle

    
    def calc_entropy(self, data): # vypočíta entropiu pre data
        p = data["failure"].value_counts(normalize=True)
        return abs((p * np.log2(p)).sum())

    def pick_best_attribute(self) -> tuple: # vyberá najlepší atribút pre aktuálny uzol
        cols = self.data.columns.to_list()
        cols.pop(cols.index("failure"))

        best_ig = 0
        best_attr = None
        best_split_point = None

        for col in cols:
            ig, split_point = self.try_attribute(col)
            if ig > best_ig:
                best_ig = ig
                best_attr = col
                best_split_point = split_point

        return best_attr, best_split_point

    def live_and_make_children(self):
        # OPRAVA: väčšinová trieda pre prípad, že uzol bude list
        self.prediction = self.data["failure"].mode().iloc[0]

        # OPRAVA: ukončovacia podmienka
        if self.entropy == 0:
            self.final = True
            return

        self.atribute, self.split_point = self.pick_best_attribute()

        # OPRAVA: žiadny atribút nepomáha -> list
        if self.atribute is None:
            self.final = True
            return

        # OPRAVA: správne indexovanie
        mask = self.data[self.atribute] <= self.split_point
        self.left_child = Node(self.data[mask])
        self.right_child = Node(self.data[~mask])
        self.left_child.live_and_make_children()
        self.right_child.live_and_make_children()

    def try_attribute(self, attribute: str) -> tuple:
        # OPRAVA: všetky atribúty (aj kategorické/one-hot kódované ako čísla)
        # sa spracujú rovnako cez prahy, aby bol výsledok vždy (ig, split_point)
        # a dal sa použiť v live_and_make_children
        temp = self.data[[attribute, "failure"]].sort_values(by=attribute)
        x = temp[attribute].to_numpy()
        y = temp["failure"].to_numpy()
        n = len(y)

        # prahy len medzi rôznymi hodnotami atribútu
        valid = x[:-1] != x[1:]
        if not valid.any():
            return 0, None

        # počty jednotiek vľavo od každého možného prahu (kumulatívne)
        ones_left = np.cumsum(y)[:-1]
        n_left = np.arange(1, n)
        n_right = n - n_left
        ones_right = y.sum() - ones_left

        def h(p):
            with np.errstate(divide="ignore", invalid="ignore"):
                t = np.where(p > 0, -p * np.log2(p), 0.0)
            return t

        # binárna entropia: h(p) + h(1-p)
        ent_left = h(ones_left / n_left) + h(1 - ones_left / n_left)
        ent_right = h(ones_right / n_right) + h(1 - ones_right / n_right)

        weighted_entropy = (n_left * ent_left + n_right * ent_right) / n
        weighted_entropy = np.where(valid, weighted_entropy, np.inf)

        i = int(np.argmin(weighted_entropy))
        info_gain = self.entropy - weighted_entropy[i]
        best_split_point = (x[i] + x[i + 1]) / 2

        return info_gain, best_split_point


In [53]:
class ID3_Classifier():
    def __init__(self):
        self.root = None

    def train(self, data):
        self.root = Node(data)
        self.root.live_and_make_children()

    def predict(self, vector):
        crossroad = self.root
        if crossroad == None: 
            print("Najprv natrénuj model: ID3_Classifier.train(dataframe)")
            return None
        while not crossroad.final:
            if vector[crossroad.atribute] <= crossroad.split_point:
                crossroad = crossroad.left_child
            else:
                crossroad = crossroad.right_child

        return crossroad.prediction

id3 = ID3_Classifier()
id3.train(train_df)

vector = train_df.iloc[10]
print(id3.predict(vector))

1
